# DML

Este notebook es una versión resumida del notebook [The effect of gun ownership on gun homicide](https://colab.research.google.com/github/CausalAIBook/MetricsMLNotebooks/blob/main/PM4/r_dml_inference_for_gun_ownership.irnb#scrollTo=u8n1149MolrR)

In [5]:
suppressWarnings(
pacman::p_load(glmnet, randomForest,xtable,dplyr,sandwich)
)

In [6]:
file <- "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/gun_clean.csv"
data <- read.csv(file)
dim(data)

[1] 3900  415

In [7]:
# Note: These data are population weighted. Specifically,
# looking at the JBES replication files, they seem to be multiplied
# by sqrt((1/T sum_t population_{j,t})/100000). To get the
# unweighted variables need to divide by this number - which we can
# get from the time effects. We are mostly just going to use the weighted
# variables as inputs - except for time and county. We'll take
# cross-sectional and time series means of these weighted variables
# as well. Note that there is nothing wrong with this, but it does not
# reproduce a weighted regression in a setting where covariates may
# enter nonlinearly and flexibly.

## County FE
county_vars <- select(data, starts_with("X_J"))

## Time variables and population weights
# Pull out time variables
time_vars <- select(data, starts_with("X_T"))

# Use these to construct population weights
pop_weights <- rowSums(time_vars)

# Unweighted time variables
time_vars <- time_vars / pop_weights

# For any columns with only zero (like the first one), just drop
time_vars <- time_vars[, colSums(time_vars != 0) > 0]

# Create time index
time_ind <- rowSums(time_vars * (seq(1:20)))

In [8]:
# Function to find variable names
var_list <- function(df = NULL, type = c("numeric", "factor", "character"), pattern = "", exclude = NULL) {
  vars <- character(0)
  if (any(type %in% "numeric")) {
    vars <- c(vars, names(df)[sapply(df, is.numeric)])
  }
  if (any(type %in% "factor")) {
    vars <- c(vars, names(df)[sapply(df, is.factor)])
  }
  if (any(type %in% "character")) {
    vars <- c(vars, names(df)[sapply(df, is.character)])
  }
  vars[(!vars %in% exclude) & grepl(vars, pattern = pattern)]
}

# census control variables
census <- NULL
census_var <- c("^AGE", "^BN", "^BP", "^BZ", "^ED", "^EL", "^HI", "^HS", "^INC", "^LF", "^LN",
                "^PI", "^PO", "^PP", "^PV", "^SPR", "^VS")

for (i in seq_along(census_var)) {
  census <- append(census, var_list(data, pattern = census_var[i]))
}

# other control variables
X1 <- c("logrobr", "logburg", "burg_missing", "robrate_missing")
X2 <- c("newblack", "newfhh", "newmove", "newdens", "newmal")

# "treatment" variable
d <- "logfssl"

# outcome variable
y <- "logghomr"

# new data frame for time index
usedata <- as.data.frame(time_ind)
colnames(usedata) <- "time_ind"
usedata[, "weights"] <- pop_weights

var_list <- c(y, d, X1, X2, census)
for (i in seq_along(var_list)) {
  usedata[, var_list[i]] <- data[, var_list[i]]
}

####################### Construct county specific means,
# time specific means, initial conditions

# Initial conditions
var_list0 <- c(y, X1, X2, census)
for (i in seq_along(var_list0)) {
  usedata[, paste(var_list0[i], "0", sep = "")] <- kronecker(
    usedata[time_ind == 1, var_list0[i]],
    rep(1, 20)
  )
}

# County means
var_list_j <- c(X1, X2, census)
county_vars <- as.matrix(county_vars)
for (i in seq_along(var_list_j)) {
  usedata[, paste(var_list_j[i], "J", sep = "")] <-
    county_vars %*% qr.solve(county_vars, as.matrix(usedata[, var_list_j[i]]))
}

# Time means
time_vars <- as.matrix(time_vars)
for (i in seq_along(var_list_j)) {
  usedata[, paste(var_list_j[i], "T", sep = "")] <-
    time_vars %*% qr.solve(time_vars, as.matrix(usedata[, var_list_j[i]]))
}

In [10]:
set.seed(123)

# Cross-fitting
n <- nrow(usedata)
Kf <- 5 # Number of cross-fitting folds
sampleframe <- rep(1:Kf, ceiling(n / Kf))
cvgroup <- sample(sampleframe, size = n, replace = FALSE) # Cross-fitting groups

# Initialize variables for cross-fit predictions
yhat_r <- matrix(NA, n, 3) # Going to consider 3 learners
dhat_r <- matrix(NA, n, 3)

# Cross-fitting loop
for (k in 1:Kf) {
  cat("fold: ", k, "\n")
  indk <- cvgroup == k

  ktrain <- usedata[!indk, ]
  ktest <- usedata[indk, ]

  #### Simple regression models ####

  lmyk_a <- lm(logghomr ~ . - logfssl, data = ktrain)
  yhat_r[indk, 1] <- ktest$logghomr - predict(lmyk_a, ktest)
  lmdk_a <- lm(logfssl ~ . - logghomr, data = ktrain)
  dhat_r[indk, 1] <- ktest$logfssl - predict(lmdk_a, ktest)

  #### Penalized Linear Models ####

  # Lasso - default CV tuning
  ytrain <- as.matrix(usedata[!indk, "logghomr"])
  dtrain <- as.matrix(usedata[!indk, "logfssl"])
  xtrain <- as.matrix(usedata[!indk, !names(usedata) %in%
                                c("logghomr", "logfssl")])
  ytest <- as.matrix(usedata[indk, "logghomr"])
  dtest <- as.matrix(usedata[indk, "logfssl"])
  xtest <- as.matrix(usedata[indk, !names(usedata) %in%
                               c("logghomr", "logfssl")])

  lassoyk <- cv.glmnet(xtrain, ytrain)
  yhat_r[indk, 2] <- ytest - predict(lassoyk, newx = xtest, s = "lambda.min")

  lassodk <- cv.glmnet(xtrain, dtrain)
  dhat_r[indk, 2] <- dtest - predict(lassodk, newx = xtest, s = "lambda.min")

  # Random forest
  rfyk <- randomForest(logghomr ~ . - logfssl, data = ktrain)
  yhat_r[indk, 3] <- ktest$logghomr - predict(rfyk, ktest)
  rfdk <- randomForest(logfssl ~ . - logghomr, data = ktrain)
  dhat_r[indk, 3] <- ktest$logfssl - predict(rfdk, ktest)
}

fold:  1 
fold:  2 
fold:  3 
fold:  4 
fold:  5 


In [11]:
# Predictions done, now DML

rmse_y <- sqrt(colMeans(yhat_r^2))
rmse_d <- sqrt(colMeans(dhat_r^2))

# dml coefficient estimates
b_dml <- rep(NA, 3)
s_dml <- rep(NA, 3)
for (k in 1:3) {
  lm_k <- lm(yhat_r[, k] ~ dhat_r[, k] - 1)
  v_k <- vcovHC(lm_k)
  b_dml[k] <- lm_k$coefficients
  s_dml[k] <- sqrt(v_k)
}


In [13]:
## Display results
table1 <- matrix(0, 3, 2)
table1[, 1] <- rmse_y
table1[, 2] <- rmse_d
colnames(table1) <- c("RMSE Y", "RMSE D")
rownames(table1) <- c(
  "OLS","Lasso (CV)", "Random Forest"
)
tab1 <- xtable(table1, digits = c(0, 4, 4))
tab1


% latex table generated in R 4.5.2 by xtable 1.8-8 package
% Fri Nov 13 17:07:42 2026
\begin{table}[ht]
\centering
\begin{tabular}{rrr}
  \hline
 & RMSE Y & RMSE D \\ 
  \hline
OLS & 173684094827.7515 & 61705048538.9272 \\ 
  Lasso (CV) & 0.4625 & 0.1353 \\ 
  Random Forest & 0.4024 & 0.1246 \\ 
   \hline
\end{tabular}
\end{table}

In [14]:

table2 <- matrix(0, 3, 2)
table2[, 1] <- b_dml
table2[, 2] <- s_dml
colnames(table2) <- c("Point Estimate", "Std. Error")
rownames(table2) <- c(
  "OLS","Lasso (CV)", "Random Forest"
)
tab2 <- xtable(table2, digits = c(0, 4, 4))
tab2

% latex table generated in R 4.5.2 by xtable 1.8-8 package
% Fri Nov 13 17:09:01 2026
\begin{table}[ht]
\centering
\begin{tabular}{rrr}
  \hline
 & Point Estimate & Std. Error \\ 
  \hline
OLS & -0.1088 & 0.0301 \\ 
  Lasso (CV) & 0.2757 & 0.0565 \\ 
  Random Forest & 0.0575 & 0.0522 \\ 
   \hline
\end{tabular}
\end{table}

In [15]:
lmA <- lm(logghomr ~ ., data = usedata)
vcA <- vcovHC(lmA)
cat("OLS All:", lmA$coefficients["logfssl"], " (", sqrt(vcA["logfssl", "logfssl"]), ")\n")

Warning message:
In meatHC(x, type = type, omega = omega) :
  HC3 covariances are numerically unstable for hat values close to 1 (and undefined if exactly 1) as for observation(s) 1


OLS All: 6.032229e+25  ( 9.785258e+42 )
